In [1]:
import os
import re
import json
from typing import Optional
from groq import Groq
from pydantic import BaseModel, ConfigDict, ValidationError, conint

In [17]:
Score = conint(ge=0, le=5)


class EvidenceItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscore: str
    quote: str
    reason: str


class MotivationSubscores(BaseModel):
    model_config = ConfigDict(extra="forbid")
    university_specificity: Score
    program_fit: Score
    goal_alignment: Score
    intrinsic_motivation: Score
    specificity_of_reasoning: Score


class LeadershipPotentialSubscores(BaseModel):
    model_config = ConfigDict(extra="forbid")
    leadership_definition_quality: Score
    concrete_example_presence: Score
    initiative: Score
    responsibility: Score
    impact: Score
    reflection: Score


class ResponseStructureSubscores(BaseModel):
    model_config = ConfigDict(extra="forbid")
    clarity: Score
    coherence: Score
    completeness: Score
    relevance: Score
    conciseness: Score


class Motivation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscores: MotivationSubscores
    evidence: list[EvidenceItem]
    weaknesses: list[str]


class LeadershipPotential(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscores: LeadershipPotentialSubscores
    evidence: list[EvidenceItem]
    weaknesses: list[str]


class ResponseStructure(BaseModel):
    model_config = ConfigDict(extra="forbid")
    subscores: ResponseStructureSubscores
    evidence: list[EvidenceItem]
    weaknesses: list[str]


class ContextNotes(BaseModel):
    model_config = ConfigDict(extra="forbid")
    family_support_context: str
    encouragement_source: str


class TranscriptScoringResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    motivation: Motivation
    leadership_potential: LeadershipPotential
    response_structure: ResponseStructure
    context_notes: ContextNotes
    risk_flags: list[str]
    missing_evidence: list[str]


def extract_json_object(text: str) -> dict:
    content = text.strip()
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\s*|\s*```$", "", content, flags=re.IGNORECASE | re.DOTALL).strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", content, flags=re.DOTALL)
        if not match:
            raise ValueError("Ответ модели не содержит корректный JSON-объект")
        return json.loads(match.group(0))


def get_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY не задан в окружении")
    return Groq(api_key=api_key)


def generate_response(prompt: str, text: str, model: Optional[str] = None) -> dict:
    if not prompt.strip():
        raise ValueError("prompt не должен быть пустым")
    if not text.strip():
        raise ValueError("text не должен быть пустым")

    client = get_client()
    model_name = model or os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": text},
            ],
            temperature=0.9,
        )
    except Exception as error:
        raise RuntimeError(f"Ошибка Groq API: {error}") from error

    answer = completion.choices[0].message.content if completion.choices else ""
    payload = extract_json_object(answer or "")

    try:
        validated = TranscriptScoringResult.model_validate(payload)
    except ValidationError as error:
        raise ValueError(f"Ответ не прошел валидацию схемы: {error}") from error

    return {
        "answer": validated.model_dump(),
        "model": model_name,
    }

In [18]:
def read_text_file(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Файл не найден: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

prompt_text = read_text_file("prompt.txt")
input_text = read_text_file("input.txt")

print("Файлы загружены.")

Файлы загружены.


In [19]:
result = generate_response(prompt=prompt_text, text=input_text, model=None)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "answer": {
    "motivation": {
      "subscores": {
        "university_specificity": 4,
        "program_fit": 5,
        "goal_alignment": 5,
        "intrinsic_motivation": 5,
        "specificity_of_reasoning": 5
      },
      "evidence": [
        {
          "subscore": "university_specificity",
          "quote": "I want to study in an environment where I can not only learn technical skills but also actually build real products.",
          "reason": "Candidate explains why inVision U's environment is a good fit, but could elaborate more on specific aspects of the university."
        },
        {
          "subscore": "program_fit",
          "quote": "I am particularly interested in the Innovative IT Product Design and Development program. The reason is that I have always been interested in technology, but not just coding itself, more like how technology can solve real problems for people.",
          "reason": "Candidate clearly explains why the program matches their in

In [20]:
result_count = {}
sections = ("motivation", "leadership_potential", "response_structure")
for section in sections:
    subscores = result["answer"][section]["subscores"]
    result_count[section] = sum(subscores.values())

result_count["total"] = sum(result_count.values())
print(json.dumps(result_count, ensure_ascii=False, indent=2))

{
  "motivation": 24,
  "leadership_potential": 29,
  "response_structure": 24,
  "total": 77
}
